In [1]:
import math
from tqdm import tqdm
import numpy as np

In [45]:
class Positions:
    def __init__(self, n = 15):
        self.positions = [(0,0,0,0,0,0)]
        self.pos2id = {(0,0,0,0,0,0): 0}
        self.size_group_starts = [0]
        
        for size in range(1, n + 1):
            self._build_next_size_positions()
            
        self.expectations = np.zeros(len(self.positions), dtype=float)
        self.comb2cubes, self.cubes2comb = self._get_cubes_combinations()
        self.best_next_positions = (-1) * np.ones((len(self), len(self.comb2cubes)), dtype='int32')
        
    
    def _get_cubes_combinations(self):
        comb2cubes = []
        for cube1 in range(1, 7):
            for cube2 in range(cube1, 7):
                comb2cubes.append((cube1, cube2))
        cubes2comb = dict()
        for i, cubes in enumerate(comb2cubes):
            cubes2comb[cubes] = i
        
        return comb2cubes, cubes2comb
        
    
    def _build_next_size_positions(self):
        # initiates inner arrays and dicts to generate the next, returns 0 in case of
        size = len(self.size_group_starts)
        size_group_start = len(self.positions)
        self.size_group_starts.append(size_group_start)
        
        prev_size_group_start = self._get_size_group_start(size - 1)
        prev_size_group_finish = size_group_start
        
        for position in self.positions[prev_size_group_start:prev_size_group_finish]:
            for i in range(5, -1, -1):
                new_position = list(position)
                new_position[i] += 1
                new_position = tuple(new_position)
                
                if new_position not in self.pos2id:
                    self.pos2id[new_position] = len(self.positions)
                    self.positions.append(new_position)
                    
        return 1
    
    
    def _get_size_group_start(self, size):
        if size >= len(self.size_group_starts):
            return None    
        return self.size_group_starts[size]
    
    
    def _get_len_of_size_group(self, size):
        size_group_start = self._get_size_group_start(size)
        if size_group_start is None:
            return 0
        
        next_size_group_start = self._get_size_group_start(size + 1)
        if next_size_group_start is None:
            return len(self.positions) - size_group_start
        
        return next_size_group_start - size_group_start
    
    
    def _apply_cube_inplace(self, position_listed, i, cube_value):
        if position_listed[i] == 0:
            return 0

        # Нельзя снимать шашку перебором, если есть шашки дальше от выхода
        if cube_value > 6 - i and sum(position_listed[0:i]) > 0:
            return 0

        # Снятие шашки ровно по кубику
        if cube_value == 6 - i:
            position_listed[i] -= 1
            return 1

        # Обычный ход внутри доски
        if cube_value < 6 - i:
            position_listed[i] -= 1
            position_listed[i + cube_value] += 1
            return 1

        # Снятие шашки перебором
        if cube_value > 6 - i:
            position_listed[i] -= 1
            return 1

        return 0
    
    
    def _revert_cube_inplace(self, position_listed, i, cube_value):
        # Если ход был снятием шашки с доски
        if cube_value >= 6 - i:
            position_listed[i] += 1
        else:
            position_listed[i + cube_value] -= 1
            position_listed[i] += 1
    
    
    def _get_next_positions_ordered_cubes(self, current_position_listed, cubes, 
                                          final_position_ids, used_cubes_number, maked_moves=0):
        # recurtionally appends two lists:
        #     "final_position_ids" with possible final positions after use all of cubes in its initial order
        #     "used_cubes" with the number of used cubes to reach this final position
        
        if maked_moves == len(cubes):
            final_position_ids.append(self.pos2id[tuple(current_position_listed)])
            used_cubes_number.append(maked_moves)
            return
        
        is_final_position = True
        for i in range(6):
            res = self._apply_cube_inplace(current_position_listed, i, cubes[maked_moves])
            if res:
                is_final_position = False
                self._get_next_positions_ordered_cubes(current_position_listed, cubes,
                                                       final_position_ids, used_cubes_number, maked_moves + 1)
                self._revert_cube_inplace(current_position_listed, i, cubes[maked_moves])
        
        if is_final_position:
            final_position_ids.append(self.pos2id[tuple(current_position_listed)])
            used_cubes_number.append(maked_moves)
        
        
    def get_legal_next_positions_ids(self, current_position, cube1, cube2):
        current_position_listed = list(current_position)
        final_position_ids = []
        used_cubes_number = []
        if cube1 == cube2:
            cubes = [cube1] * 4
            self._get_next_positions_ordered_cubes(current_position_listed, cubes,
                                                   final_position_ids, used_cubes_number)
        else:
            cubes = [cube1, cube2]
            self._get_next_positions_ordered_cubes(current_position_listed, cubes,
                                                   final_position_ids, used_cubes_number)
            cubes = [cube2, cube1]
            self._get_next_positions_ordered_cubes(current_position_listed, cubes,
                                                   final_position_ids, used_cubes_number)
            
        max_length = max(used_cubes_number)
        final_position_ids = set([final_position_ids[i] for i in range(len(final_position_ids)) 
                                  if used_cubes_number[i] == max_length])
        return list(final_position_ids)
    
    
    def get_legal_next_positions(self, current_position, cube1, cube2):
        final_position_ids = self.get_legal_next_positions_ids(current_position, cube1, cube2)
        final_positions = [self.positions[i] for i in final_position_ids]
        return final_positions
    
    
    def _get_best_next_position_id(self, current_position, cube1, cube2):
        if cube1 > cube2:
            cube1, cube2 = cube2, cube1

        current_position_id = self.pos2id[current_position]
        comb_id = self.cubes2comb[(cube1, cube2)]

        cached = self.best_next_positions[current_position_id, comb_id]
        if cached != -1:
            return cached

        legal_next_position_ids = self.get_legal_next_positions_ids(current_position, cube1, cube2)

        best_pos_id = legal_next_position_ids[0]
        best_exp = self.expectations[best_pos_id]

        for pos_id in legal_next_position_ids[1:]:
            if self.expectations[pos_id] < best_exp:
                best_exp = self.expectations[pos_id]
                best_pos_id = pos_id

        self.best_next_positions[current_position_id, comb_id] = best_pos_id
        return best_pos_id
    
    
    def get_best_next_position(self, current_position, cube1, cube2):
        best_position_id = self._get_best_next_position_id(current_position, cube1, cube2)
        return self.positions[best_position_id]


    def compute_expectations(self):
        self.expectations[0] = 0.0

        for pos_id in tqdm(range(1, len(self.positions))):
            current_position = self.positions[pos_id]
            exp_value = 0.0

            for cube1, cube2 in self.comb2cubes:
                best_next_pos_id = self._get_best_next_position_id(current_position, cube1, cube2)

                if cube1 == cube2:
                    prob = 1.0 / 36.0
                else:
                    prob = 1.0 / 18.0
                
                assert self.expectations[best_next_pos_id] != -1
                exp_value += prob * (1.0 + self.expectations[best_next_pos_id])

            self.expectations[pos_id] = exp_value
    
    
    def __len__(self):
        return len(self.positions)

In [46]:
%%time

positions = Positions(15)

Wall time: 197 ms


In [47]:
cur_pos = (6, 0, 0, 0, 0, 1)
cube1 = 2
cube2 = 2
positions.get_legal_next_positions(cur_pos, cube1, cube2)

[(3, 0, 2, 0, 1, 1),
 (2, 0, 4, 0, 0, 1),
 (4, 0, 0, 0, 2, 1),
 (4, 0, 1, 0, 0, 1)]

In [48]:
positions.compute_expectations()

100%|███████████████████████████████████████████████████████████████████████████| 54263/54263 [01:23<00:00, 650.52it/s]


In [49]:
for i in range(len(positions.positions) - 200, len(positions.positions)):
    print(positions.positions[i], positions.expectations[i])

(1, 9, 4, 1, 0, 0) 9.731738397143397
(2, 8, 4, 1, 0, 0) 9.738681816833648
(3, 7, 4, 1, 0, 0) 9.77378912870179
(4, 6, 4, 1, 0, 0) 9.842806791128412
(5, 5, 4, 1, 0, 0) 9.940388397272672
(6, 4, 4, 1, 0, 0) 10.058585323299662
(7, 3, 4, 1, 0, 0) 10.195407011221485
(8, 2, 4, 1, 0, 0) 10.355070579179515
(9, 1, 4, 1, 0, 0) 10.546138485513179
(10, 0, 4, 1, 0, 0) 10.776524224291169
(0, 11, 3, 1, 0, 0) 9.891470290913553
(1, 10, 3, 1, 0, 0) 9.883775988522101
(2, 9, 3, 1, 0, 0) 9.886007811005074
(3, 8, 3, 1, 0, 0) 9.914007319341241
(4, 7, 3, 1, 0, 0) 9.974741652086417
(5, 6, 3, 1, 0, 0) 10.06392998806021
(6, 5, 3, 1, 0, 0) 10.17396745669449
(7, 4, 3, 1, 0, 0) 10.298683328546321
(8, 3, 3, 1, 0, 0) 10.439310089414839
(9, 2, 3, 1, 0, 0) 10.60234498878668
(10, 1, 3, 1, 0, 0) 10.7958036750357
(11, 0, 3, 1, 0, 0) 11.027761167530821
(0, 12, 2, 1, 0, 0) 10.072636965960578
(1, 11, 2, 1, 0, 0) 10.061190898082879
(2, 10, 2, 1, 0, 0) 10.056988068390364
(3, 9, 2, 1, 0, 0) 10.075598106925273
(4, 8, 2, 1, 0, 0) 1

In [102]:
step = 1
current_position = (3, 3, 3, 2, 2, 2)
while current_position != (0, 0, 0, 0, 0, 0):
    dice1 = np.random.randint(1, 7)
    dice2 = np.random.randint(1, 7)
    if dice1 < dice2:
        dice1, dice2 = dice2, dice1
    current_position_id = positions.pos2id[current_position]
    exp = positions.expectations[current_position_id]
    print(f"{step}) \t {current_position}\t{dice1}, {dice2}\t{exp}")
    current_position = positions.get_best_next_position(current_position, dice1, dice2)
    step += 1

1) 	 (3, 3, 3, 2, 2, 2)	3, 1	8.13892320259791
2) 	 (3, 3, 3, 1, 2, 1)	2, 1	7.498123978307827
3) 	 (3, 3, 3, 1, 1, 0)	5, 2	7.003144054238832
4) 	 (3, 2, 3, 1, 0, 0)	6, 2	6.105775199542388
5) 	 (2, 2, 2, 1, 1, 0)	1, 1	5.130629691057204
6) 	 (2, 2, 1, 1, 1, 0)	6, 1	4.626198320121636
7) 	 (1, 1, 2, 1, 1, 0)	3, 2	3.7798780933225
8) 	 (1, 1, 2, 0, 0, 0)	6, 1	3.036121844017021
9) 	 (0, 1, 1, 1, 0, 0)	5, 1	2.1608796296296298
10) 	 (0, 0, 1, 0, 1, 0)	6, 1	1.361111111111111
11) 	 (0, 0, 0, 0, 0, 1)	6, 2	1.0000000000000002


In [67]:
import sys

In [42]:
np.random.randint(1, 3)

1